# Parameter axis sweep

## Load the simulations

In [1]:
from ipynb_boilerplate import *
AGG = True
if AGG: configure_matplotlib(backend='Agg')
SAVE = True

PARAM_KEY = 'sr'
PARAM_KEY_TEX = 's_r'
PARAMS_FIXED = SYSTEM_A_REFERENCE.remove(PARAM_KEY)

simulations_batch = GridSimulationFromNPZ.dict_from_dir_paths(
    PARAM_KEY, 
    SIM_DIR_PATHS,
    ('c', 's'),
    ('f', 'mD', 'mC', 'uRMS'),
    PARAMS_FIXED,
    lazy=(not AGG),
    sorting=lambda d: dict(sorted(d.items(), reverse=False)),
)
save_fig = partial(
    save_figure, 
    dir_path=DIR_FIGS, 
    prefix=PARAM_KEY, 
    pickle=True,
    close=AGG,
    file_ext=('svg', 'pdf', 'png'),
) if SAVE else pass_anything

Ly = 1.0
Lx = Ly * PARAMS_FIXED['aspect']

In [7]:
print('Before parameter batching')
for i in SIM_DIR_PATHS: print(i)
print('After parameter batching')
for i in simulations_batch.values(): print(i.dir_path)

Before parameter batching
./data__c_stabilization=None|c_limits=True/t_stop=120.0__Ra=100.0|Da=1.0|epsilon=0.01|zeta0=0.9|sr=0.2|cr=0.0|aspect=2.0|Nx=160|Ny=160__8cdc0d7deee27009/
./data__c_stabilization=None|c_limits=True/t_stop=120.0__Ra=100.0|Da=10.0|epsilon=0.01|zeta0=0.9|sr=0.2|cr=0.0|aspect=2.0|Nx=160|Ny=160__f1e8b107b29abbbb/
./data__c_stabilization=None|c_limits=True/t_stop=120.0__Ra=100.0|Da=100.0|epsilon=0.01|zeta0=0.9|sr=0.2|cr=0.0|aspect=2.0|Nx=160|Ny=160__5e822ff46ee05fa4/
./data__c_stabilization=None|c_limits=True/t_stop=120.0__Ra=100.0|Da=1000.0|epsilon=0.01|zeta0=0.9|sr=0.2|cr=0.0|aspect=2.0|Nx=160|Ny=160__3823d90f6e03a6fd/
./data__c_stabilization=None|c_limits=True/t_stop=120.0__Ra=500.0|Da=1.0|epsilon=0.01|zeta0=0.9|sr=0.2|cr=0.0|aspect=2.0|Nx=160|Ny=160__795c9d7125905bce/
./data__c_stabilization=None|c_limits=True/t_stop=120.0__Ra=500.0|Da=10.0|epsilon=0.01|zeta0=0.9|sr=0.2|cr=0.0|aspect=2.0|Nx=160|Ny=160__c4deca4f8fe27111/
./data__c_stabilization=None|c_limits=True/

## Select the simulations

In [3]:
prm_targets = None
simulations = {
    k: v for k, v in simulations_batch.items()
    if include_prm(k, prm_targets)
}

In [4]:
print('After parameter selection')
for i in simulations.values(): print(i.dir_path)

After parameter selection
./data__c_stabilization=None|c_limits=True/t_stop=120.0__Ra=1000.0|Da=1.0|epsilon=0.01|zeta0=0.9|sr=0.2|cr=0.0|aspect=2.0|Nx=200|Ny=200__9563b4a91df22147/
./data__c_stabilization=None|c_limits=True/t_stop=120.0__Ra=1000.0|Da=10.0|epsilon=0.01|zeta0=0.9|sr=0.2|cr=0.0|aspect=2.0|Nx=200|Ny=200__6960ce4014553e45/
./data__c_stabilization=None|c_limits=True/t_stop=120.0__Ra=1000.0|Da=100.0|epsilon=0.01|zeta0=0.9|sr=0.2|cr=0.0|aspect=2.0|Nx=200|Ny=200__34ee6f0aa42efdfd/
./data__c_stabilization=None|c_limits=True/t_stop=120.0__Ra=1000.0|Da=1000.0|epsilon=0.01|zeta0=0.9|sr=0.2|cr=0.0|aspect=2.0|Nx=200|Ny=200__476f009e5105f01c/


In [ ]:
function_series = 'c'
for param_value, sim in simulations.items():
    time_series = sim[function_series].time_series
    print(f'{PARAM_KEY} = {param_value}')
    print(f"n_snapshots = {len(time_series)} t_min = {np.min(time_series)}  t_max = {np.max(time_series)}")

## Flux, root-mean-square velocity & mass

In [ ]:
HLINES = (not PARAM_KEY in ('sr', 'cr', 'epsilon', 'zeta0'))
legend_labels = []
f_lines, uRMS_lines = [], []
mC_lines, mD_lines, mC_frac_lines, mD_frac_lines = [], [], [], []

if HLINES:
    epsilon, zeta0, sr, cr = PARAMS_FIXED['epsilon', 'zeta0', 'sr', 'cr']
    mC_initial = mass_capillary_initial(epsilon, zeta0, sr, Lx, Ly)
    mD_initial = mass_dissolved_initial(zeta0, sr, cr, Lx, Ly)
    m_initial = mC_initial + mD_initial
    mC_asymp = mass_capillary_asymptote(m_initial, epsilon, Lx, Ly)
    mD_asymp = mass_dissolved_asymptote(m_initial, epsilon, Lx, Ly)

for param_value, sim in simulations.items():
    legend_labels.append(as_int_if_close(param_value))
    f, mC, mD, uRMS = sim['f', 'mC', 'mD', 'uRMS']
    fZeta0, fZetaPlus, fZetaMinus = f.split()
    f_lines.append((fZeta0.time_series, [np.sum(i) for i in fZeta0.value_series]))
    uRMS_lines.append((uRMS.time_series, uRMS.value_series))
    mC_lines.append((mC.time_series, mC.value_series))
    mD_lines.append((mD.time_series, mD.value_series))
    mC_frac_lines.append((mC.time_series, [i / (i + j) for i, j in zip(mC.value_series, mD.value_series)]))
    mD_frac_lines.append((mD.time_series, [j / (i + j) for i, j in zip(mC.value_series, mD.value_series)]))
    
    
line_kws = dict(
    cyc='black',
    x_label='$t$',
    legend_labels=legend_labels,
    legend_title=f'${PARAM_KEY_TEX}$',
)
hline_kws = dict(
    xmin=0, xmax=120.0, 
    linestyles=(5, (10, 3)), colors='gray', linewidths=0.75,
)

fig, ax = plot_line(f_lines, y_label='$F_{y=\zeta_0}$', x_lims=(0, 10.0), **line_kws)
save_fig('fZeta(t)_early')(fig)

fig, ax = plot_line(f_lines, y_label='$F_{y=\zeta_0}$', x_lims=(0, 60.0), **line_kws)
save_fig('fZeta(t)_late')(fig)

fig, ax = plot_line(uRMS_lines, y_label='$\mathrm{rms}(\mathbf{u})$', **line_kws)
ax.set_yscale('log')
save_fig('uRMS(t)')(fig)

fig, ax = plot_line(mC_lines, y_label='$m_C$', **line_kws)
if HLINES: ax.hlines([mC_initial, mC_asymp], **hline_kws)
save_fig('mC(t)')(fig)

fig, ax = plot_line(mD_lines, y_label='$m_D$', **line_kws)
if HLINES: ax.hlines([mD_initial, mD_asymp], **hline_kws)
save_fig('mD(t)')(fig)

fig, ax = plot_line(mC_frac_lines, y_label='$m_C/(m_C + m_D)$', **line_kws)
save_fig('mC(t)_fraction')(fig)

fig, ax = plot_line(mD_frac_lines, y_label='$m_D/(m_C + m_D)$', **line_kws)
save_fig('mD(t)_fraction')(fig)

## Horizontal averages

In [ ]:
slc = slice(0, None, 10)
titles = []
sAvgX_lines, sAvgX_legend_labels = [], []
cAvgX_lines, cAvgX_legend_labels = [], []

def sAvgX_pipeline(
    s: GridFunction, 
    zeta0: float | None,
    y_resample: int | None = 20,
) -> GridFunction:
    """
    Correcting the `DP₀ -> P₁` interpolation error at `y = ζ₀` 
    such that `s(x, y = ζ₀) = 0`.
    """
    if y_resample is None:
        s_fine = s
    else:
        s_fine = resample_grid(s, (1, y_resample))
    sAvgX = average_grid(use_cache=True)(s_fine, 'x')
    if zeta0: 
        return copy_grid(sAvgX, (zeta0, ), 0.0)
    else:
        return sAvgX


for param_value, sim in simulations.items():
    titles.append(f'\n${PARAM_KEY_TEX}={as_int_if_close(param_value)}$')
    s, c, zeta0 = sim['s', 'c', 'zeta0']
    sAvgX = GridFunctionSeries(
        [sAvgX_pipeline(i, zeta0)for i in s.series[slc]], 
        s.time_series[slc], 
        'sAvgX',
    )
    sAvgX_legend_labels.append((min(sAvgX.time_series), max(sAvgX.time_series)))
    sAvgX_lines.append(sAvgX.series)
    cAvgX = GridFunctionSeries(
        [average_grid(use_cache=True)(i, 'x') for i in c.series[slc]], c.time_series[slc], 'cAvgX',
    )
    cAvgX_legend_labels.append((min(cAvgX.time_series), max(cAvgX.time_series)))
    cAvgX_lines.append(cAvgX.series)


multiline_kws = dict(
    cyc='jet',
    title=titles, 
    legend_title='$t$',
    flip=True,
    y_label='$y$',
)

fig, *_  = plot_line_multifigure(n_cols=1, cbars=True)(
    cAvgX_lines, 
    legend_labels=cAvgX_legend_labels,
    x_label='$\langle c\\rangle_x$', 
    y_lims=(0, Ly),
    **multiline_kws,
)
save_fig('cAvgX(t)')(fig)

fig, *_  = plot_line_multifigure(n_cols=1, cbars=True)(
    sAvgX_lines, 
    legend_labels=sAvgX_legend_labels,
    x_label='$\langle s\\rangle_x$', 
    y_lims=(zeta0 * Ly, Ly),
    **multiline_kws,
)
save_fig('sAvgX(t)')(fig)

## Subdomain averages

In [6]:
legend_labels = []
cPlus_lines, sPlus_lines, cMinus_lines = [], [], []

for param_value, sim in simulations.items():
    legend_labels.append(as_int_if_close(param_value))
    s, c = sim['s', 'c']
    y_axis = c.mesh.y_axis
    zeta0_index = as_index(y_axis, zeta0)
    slcPlus = slice(zeta0_index, None)
    slcMinus = slice(0, zeta0_index)
    cPlus = NPyConstantSeries(
        average_grid(c.series, ('x', 'y'), (':', slcPlus)), c.time_series, 'cPlus',
    )
    sPlus = NPyConstantSeries(
        average_grid(s.series, ('x', 'y'), (':', slcPlus)), s.time_series, 'sPlus',
    )
    cMinus = NPyConstantSeries(
        average_grid(c.series, ('x', 'y'), (':', slcMinus)), c.time_series, 'cMinus',
    )
    cPlus_lines.append((cPlus.time_series, cPlus.value_series))
    sPlus_lines.append((sPlus.time_series, sPlus.value_series))
    cMinus_lines.append((cMinus.time_series, cMinus.value_series))

line_kws = dict(
    cyc='black',
    x_label='$t$',
    legend_labels=legend_labels,
    legend_title=f'${PARAM_KEY_TEX}$',
)

fig, ax = plot_line(
    cPlus_lines,
    y_label='$c^+$',
    **line_kws,
)
save_fig('cPlus(t)')(fig)

fig, ax = plot_line(
    sPlus_lines,
    y_label='$s^+$',
    **line_kws,
)
save_fig('sPlus(t)')(fig)

fig, ax = plot_line(
    cMinus_lines,
    y_label='$c^-$',
    **line_kws,
)
save_fig('cMinus(t)')(fig)

## Visualization

In [7]:
cmap_file_ext = ('pdf', 'png')
c_t_targets = (5.0, 10.0, 20.0, 100.0)
s_t_targets = c_t_targets
s_crop = (zeta0 * Ly, Ly)
s_resample = (1, 10)

for c_t_trg, s_t_trg in zip(c_t_targets, s_t_targets):
    c_funcs, c_titles = [], []
    s_funcs, s_titles = [], []

    for param_value, sim in simulations.items():
        param_value = as_int_if_close(param_value)
        c = sim['c']
        c_time_index = as_index(c.time_series, c_t_trg)
        c_funcs.append(c.series[c_time_index])
        c_titles.append(f'${PARAM_KEY_TEX}={param_value}$\n $c(t={c.time_series[c_time_index]:.3f})$')
        s = sim['s']
        s_time_index = as_index(s.time_series, s_t_trg)
        s_func = resample_grid(
            crop_grid(s.series[s_time_index], (None, s_crop), use_cache=True),
            s_resample,
        )
        s_funcs.append(s_func)
        s_titles.append(f'${PARAM_KEY_TEX}={param_value}$\n $s(t={s.time_series[s_time_index]:.3f})$')

    fig, *_  = plot_colormap_multifigure(n_cols=1, cbars=(0, 1))(
        c_funcs, title=c_titles,
    )
    save_fig(f'c(x,y,t={c_t_trg})')(fig, file_ext=cmap_file_ext)

    s_cbars = True if PARAM_KEY == 'sr' else (0, sr)
    fig, *_  = plot_colormap_multifigure(n_cols=1, cbars=s_cbars)(
        s_funcs, title=s_titles, y_lims=s_crop, aspect='auto',
    )
    save_fig(f's(x,y,t={s_t_trg})')(fig, file_ext=cmap_file_ext)